In [86]:
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
import pickle

In [87]:
data = pd.read_csv("portas logicas/problemAND.csv")
data

,x0,x1,y
0,-1,-1,-1
1,-1,1,-1
2,1,-1,-1
3,1,1,1


In [88]:
x = data.iloc[:, :-1] 
y = data.iloc[:, -1] 
x_portas = np.array(x)
y_portas = np.array(y)
y_portas.shape

(4,)

In [89]:
x = np.load("CARACTERES COMPLETO/X.npy")
x = x.reshape(x.shape[0], -1)
Y = np.load("CARACTERES COMPLETO/Y_classe.npy")
Y.shape[1]
x.shape[1]

120

In [90]:
Y

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(1326, 26), dtype=float32)

In [ ]:
class NnModel:
    def __init__(self, x: np.ndarray, y: np.ndarray, hidden_neurons: int = 64, output_neurons: int = 26):
        self.hidden_neurons = hidden_neurons
        self.x = x
        self.y = y
        self.output_neurons = output_neurons
        #Captura o número de colunas(variáveis) para o modelo
        self.input_neurons = self.x.shape[1]

        #Incialização dos Pesos e Viés
        self.w1 = np.random.randn(self.input_neurons, self.hidden_neurons)
        self.b1 = np.zeros((1, self.hidden_neurons))
        self.w2 = np.random.randn(self.hidden_neurons, self.output_neurons)
        self.b2 = np.zeros((1, self.output_neurons))
        print(f'W1: {self.w1.shape}')
        print(f'B1: {self.b1.shape}')
        print(f'W2: {self.w2.shape}')
        print(f'B2: {self.b2.shape}')

        self.z1 = 0
        self.f1 = 0

    #Passo os valores de x como parâmetro, dado que quero utilizar a função do feedfoward para testar, enquanto o modelo self.x utilizo para treino
    def feedfoward(self, x: np.ndarray):
        #Equação da reta
        self.z1 = x.dot(self.w1) + self.b1

        #Função de Ativação dos neurônios de entrada para os neurônios ocultos
        self.f1 = np.tanh(self.z1)
        
        #Equação da reta (hidden layer)
        z2 = self.f1.dot(self.w2) + self.b2

        #Softmax para poder calcular as "probabilidades" de cada classe
        softmax = np.exp(z2)/np.sum(np.exp(z2), axis = 1, keepdims=True)
        return softmax

    def loss_function(self, softmax):
        #Aplicação da função de Cross Entropy para calcular o erro da rede neural
        losses = []
        for i in range(len(self.y)):
            y_pred = softmax[i, self.y[i]]
            losses.append(-np.log(y_pred + 1e-8))  # Evitar log(0)
        return np.mean(losses)

    def backpropation(self, softmax: np.ndarray, learning_rate: float):
        # DE -> derivada do erro
        # Dy_pred(\hat y) -> derivada do valor predito pela softmax na output layer
        # DZ2 -> derivada da equação de reta da hidden layer para a output layer
        # DW2 -> derivada dos pesos da hidden layer para a output layer
        # Df1 -> derivada da função de ativação da hidden layer (Tangente hiperbólica)
        # DZ1 -> derivada da equação de reta da input layer para a hidden layer
        # DW1 -> derivada dos pesos de entrada para a hidden layer
       
        delta2 = np.copy(softmax)
        delta2[range(self.y.shape[0]), self.y] -= 1
        dW2 = self.f1.T.dot(delta2)
        dB2 = np.sum(delta2, axis = 0, keepdims= True)

        delta1 = delta2.dot(self.w2.T) * (1 - np.power(np.tanh(self.z1) , 2))
        dW1 = self.x.T.dot(delta1)
        dB1 = np.sum(delta1, axis=0, keepdims=True)

        self.w1 +=  -learning_rate * dW1
        self.w2 += - learning_rate * dW2

        self.b1 += -learning_rate * dB1
        self.b2 += -learning_rate * dB2


    def fit(self, epochs, learning_rate: float):
        
        for epoch in range(epochs):
            output = self.feedfoward(self.x)
            loss = self.loss_function(output)
            self.backpropation(output, learning_rate)

            #Acurácia
            predictions = np.argmax(output, axis = 1)
            correct = (predictions == self.y).sum()
            accuracy = correct / len(self.y)

            if(epoch + 1) % (epochs//10) == 0:
                print(f'Epoch: [{epoch + 1} / {epochs}] Accuracy: [{accuracy:.3f}] Loss: [{loss:.3f}]')

        return predictions

In [92]:
y_indices = np.argmax(Y, axis=1)
y_indices

array([ 0,  1,  2, ..., 23, 24, 25], shape=(1326,))

In [93]:
# Preparar os dados para o modelo
y_indices = np.argmax(Y, axis=1)

# Dividir em treino e teste
np.random.seed(42)
n_samples = len(x)
indices = np.arange(n_samples)
np.random.shuffle(indices)

# 80% treino, 20% teste
split_index = int(0.8 * n_samples)
train_indices = indices[:split_index]
test_indices = indices[split_index:]

x_train = x[train_indices]
x_test = x[test_indices]
y_train = y_indices[train_indices]
y_test = y_indices[test_indices]

# Instanciar o modelo com dados de treino
model = NnModel(x_train, y_train, hidden_neurons=64, output_neurons=26)

# Treinar o modelo
predictions_train = model.fit(epochs=500, learning_rate=0.01)

# Avaliar no conjunto de teste
predictions_test = np.argmax(model.feedfoward(x_test), axis=1)
accuracy_train = (predictions_train == y_train).sum() / len(y_train)
accuracy_test = (predictions_test == y_test).sum() / len(y_test)

print("Treinamento concluído!")
print(f"Acurácia no treino: {accuracy_train:.3f}")
print(f"Acurácia no teste: {accuracy_test:.3f}")
print(f"Predições no teste: {predictions_test}")

W1: (120, 64)
B1: (1, 64)
W2: (64, 26)
B2: (1, 26)
Epoch: [50 / 500] Accuracy: [0.490] Loss: [6.782]
Epoch: [100 / 500] Accuracy: [0.927] Loss: [0.433]
Epoch: [150 / 500] Accuracy: [0.989] Loss: [0.054]
Epoch: [200 / 500] Accuracy: [0.992] Loss: [0.033]
Epoch: [250 / 500] Accuracy: [0.992] Loss: [0.024]
Epoch: [300 / 500] Accuracy: [0.993] Loss: [0.019]
Epoch: [350 / 500] Accuracy: [0.994] Loss: [0.017]
Epoch: [400 / 500] Accuracy: [0.995] Loss: [0.015]
Epoch: [450 / 500] Accuracy: [0.996] Loss: [0.013]
Epoch: [500 / 500] Accuracy: [0.996] Loss: [0.012]
Treinamento concluído!
Acurácia no treino: 0.996
Acurácia no teste: 0.763
Predições no teste: [21  8 23  9  1 16  0  8 24  5 25 23  1 10 17 17  0 24  1 21 11 21  5  1
  1  3 11 23 15  5 18  8 20  4 20 15 10 19  3 22 25 15 23 25 14  6  1 21
 22 25  7  8  0 20 10  9 12 17 18  5  2  7 10  6  2  4 16  6 11  4 18 20
  2 16 12 18 19  9 21  2 10 13 11 19 15 25 19  7 25 10 18 16  3 12 12  0
  8 20  6 11 17 20  8 22 14 15 25  0 16 18 13 17  5 24

## Backpropagation

$$\frac{\partial E}{\partial W1} = \frac{\partial E}{\partial \hat y} * \frac{\partial \hat y}{\partial Z2} * \frac{\partial Z2}{\partial W2} * \frac{\partial W2}{\partial f1} * \frac{\partial f1}{\partial Z1} * \frac{\partial Z1}{\partial W1} $$

$$ \delta_2 = \frac{\partial E}{\partial \hat y} * \frac{\partial \hat y}{\partial Z2}$$

Partindo de $\hat y$:
$$\hat y =\frac{e^{z_{21}}}{e^{z_{21}} + e^{z_{22}}}$$

Temos que $\frac{\partial \hat y}{\partial Z2}$ é:
$$u = e^{z_{21}}  \space \space \space \space \space \space v = e^{z_{21}} + e^{z_{22}}$$


$$\hat y = \frac{u}{v}$$
$$
\frac{\partial{\frac{u}{v}}}{\partial{Z2}}
=
\frac{u'v - uv'}{v^2} = \frac{
e^{z_{21}}(e^{z_{21}}+e^{z_{22}})
-
e^{z_{21}}e^{z_{21}}
}{
(e^{z_{21}}+e^{z_{22}})^2
}
=
\frac{
e^{z_{21}}e^{z_{22}}
}{
(e^{z_{21}}+e^{z_{22}})^2
}
$$
Agora calculamos:

$$
1-\hat y
$$

Substituindo:

$$
1-
\frac{e^{z_{21}}}
{e^{z_{21}}+e^{z_{22}}}
$$

Colocando no mesmo denominador:

$$
=
\frac{
e^{z_{21}}+e^{z_{22}}-e^{z_{21}}
}{
e^{z_{21}}+e^{z_{22}}
}
$$

Simplificando:

$$
1-\hat y
=
\frac{
e^{z_{22}}
}{
e^{z_{21}}+e^{z_{22}}
}
$$

Agora fazemos:

$$
\hat y(1-\hat y)
$$

Substituindo:

$$
=
\frac{e^{z_{21}}}
{e^{z_{21}}+e^{z_{22}}}
\cdot
\frac{e^{z_{22}}}
{e^{z_{21}}+e^{z_{22}}}
$$

Multiplicando:

$$
=
\frac{
e^{z_{21}}e^{z_{22}}
}{
(e^{z_{21}}+e^{z_{22}})^2
}
$$

Que é exatamente a derivada encontrada anteriormente. Assim temos que:
$$\frac{\partial \hat y}{\partial z_{21}} = \hat y(1-\hat y)$$


Agora vamos calcular a $\frac{\partial E}{\partial \hat y}$:
$$-Y'*\ln{\hat y} + -Y * \ln{\hat y}'$$

$$ \delta_2 = (0*\ln{\hat y} + (-Y * \frac{1}{\hat y})) * \frac{\partial \hat y}{\partial Z2} = (-Y * \frac{1}{\hat y}) * \hat y(1-\hat y) = -Y * (1- \hat y)  = \hat y - 1  $$
---
Cálculo das derivadas parciais da layer de entrada para a hidden layer

$$\frac{\partial E}{\partial W1} = \delta2 * W2 * \frac{\partial f1}{\partial Z1} * \frac{\partial Z1}{\partial W1}$$

$$\tanh = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}$$
$$\frac{\partial f1}{\partial Z1} =\frac{e^{z} - e^{-z}}{e^{z} + e^{-z}} = \frac{(e^{z} - e^{-z})(e^{z} + e^{-z}) - (e^{z} - e^{-z})(e^{z} + e^{-z})}{(e^{z} + e^{-z})²} = \frac{(e^{z} + e^{-z})² - (e^{z} - e^{-z})²}{(e^{z} + e^{-z})²} = 1 - \tanh²z $$
$$\frac{\partial Z1}{\partial W1} = X$$

$$\frac{\partial E}{\partial W1} = \delta2 * W2 * (1 - \tanh²z) *X$$
